# EDA y Limpieza — application_train

Este notebook documenta todas las decisiones de limpieza sobre `application_train.parquet`.  
Cada sección tiene:
- Una celda **Markdown** explicando el criterio de la decisión.
- Una celda de **código** que ejecuta la eliminación o transformación.

Al final del notebook se imprime un resumen del antes y después.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

FILENAME = "application_train.parquet"

def find_in_project(filename: str) -> Path:
    # Busca `data/<archivo>` o `Proyecto_Integrador/data/<archivo>` desde la carpeta actual
    bases = [Path.cwd(), *Path.cwd().parents]
    for base in bases:
        for d in (base / "data", base / "Proyecto_Integrador" / "data"):
            p = d / filename
            if p.exists():
                return p
    raise FileNotFoundError(
        f"No se encontró {filename}. Esperado en data/ o Proyecto_Integrador/data/ desde {Path.cwd()}"
    )

PARQUET_PATH = find_in_project(FILENAME)
app = pd.read_parquet(PARQUET_PATH)

print("CWD:", Path.cwd())
print("Leyendo:", PARQUET_PATH)
print(f"Shape original: {app.shape}")
print(f"Default rate  : {app['TARGET'].mean():.4f}")

# Guardar conteo original para el resumen final
cols_originales = app.shape[1]
log_eliminaciones = []

CWD: c:\Users\camil\OneDrive\Imágenes\PI\Proyecto_Integrador\eda
Leyendo: c:\Users\camil\OneDrive\Imágenes\PI\Proyecto_Integrador\data\application_train.parquet
Shape original: (307511, 122)
Default rate  : 0.0807


---
## 1. Eliminación: bloque housing MODE y MEDI

**Criterio: redundancia — correlación > 0.97 con _AVG**

El bloque housing describe características físicas del edificio donde vive el solicitante (área, pisos, ascensores, etc.).  
Cada característica viene en **tres versiones**: `_AVG`, `_MODE` y `_MEDI`, que son tres formas distintas de agregar el mismo dato.  
La correlación entre ellas supera 0.97 en todos los casos — prácticamente la misma información triplicada.

**Decisión:** conservar solo `_AVG` y eliminar `_MODE` y `_MEDI`.  
Esto reduce 28 columnas sin perder información.



In [3]:
housing_base = [
    'APARTMENTS', 'BASEMENTAREA', 'YEARS_BEGINEXPLUATATION', 'YEARS_BUILD',
    'COMMONAREA', 'ELEVATORS', 'ENTRANCES', 'FLOORSMAX', 'FLOORSMIN',
    'LANDAREA', 'LIVINGAPARTMENTS', 'LIVINGAREA', 'NONLIVINGAPARTMENTS', 'NONLIVINGAREA'
]

cols_housing_drop = []
for base in housing_base:
    for suffix in ['_MODE', '_MEDI']:
        col = f'{base}{suffix}'
        if col in app.columns:
            cols_housing_drop.append(col)

app = app.drop(columns=cols_housing_drop)
log_eliminaciones.append({'grupo': 'Housing MODE y MEDI', 'n': len(cols_housing_drop), 'criterio': 'Correlación >0.97 con _AVG'})

print(f'Columnas eliminadas : {len(cols_housing_drop)}')
print(f'Shape actual        : {app.shape}')

Columnas eliminadas : 28
Shape actual        : (307511, 94)


---
## 2. Eliminación: FLAG_DOCUMENT con tasa de presentación < 0.5%

**Criterio: varianza insuficiente para discriminar**

Hay 20 columnas `FLAG_DOCUMENT_*` que indican si el cliente presentó cada documento.  
La mayoría tiene una tasa de presentación extremadamente baja — menos del 0.5% de los solicitantes las entregaron.  
Una variable que es casi siempre 0 no tiene varianza suficiente para que un modelo aprenda algo de ella.

**Decisión:** eliminar todos los documentos con tasa < 0.5%.  
Se conservan los que tienen tasa suficiente: `FLAG_DOCUMENT_3` (71%), `FLAG_DOCUMENT_6` (8.8%), `FLAG_DOCUMENT_8` (8.1%), `FLAG_DOCUMENT_5` (1.5%), `FLAG_DOCUMENT_16` (1%), `FLAG_DOCUMENT_18` (0.8%).

In [4]:
doc_cols = [c for c in app.columns if c.startswith('FLAG_DOCUMENT')]
low_doc  = [c for c in doc_cols if app[c].mean() < 0.005]

print('Documentos con tasa < 0.5% (a eliminar):')
for c in low_doc:
    print(f'  {c}: {app[c].mean()*100:.3f}%')

app = app.drop(columns=low_doc)
log_eliminaciones.append({'grupo': 'FLAG_DOCUMENT tasa <0.5%', 'n': len(low_doc), 'criterio': 'Varianza insuficiente'})

print(f'\nColumnas eliminadas : {len(low_doc)}')
print(f'Shape actual        : {app.shape}')

Documentos con tasa < 0.5% (a eliminar):
  FLAG_DOCUMENT_2: 0.004%
  FLAG_DOCUMENT_4: 0.008%
  FLAG_DOCUMENT_7: 0.019%
  FLAG_DOCUMENT_9: 0.390%
  FLAG_DOCUMENT_10: 0.002%
  FLAG_DOCUMENT_11: 0.391%
  FLAG_DOCUMENT_12: 0.001%
  FLAG_DOCUMENT_13: 0.353%
  FLAG_DOCUMENT_14: 0.294%
  FLAG_DOCUMENT_15: 0.121%
  FLAG_DOCUMENT_17: 0.027%
  FLAG_DOCUMENT_19: 0.060%
  FLAG_DOCUMENT_20: 0.051%
  FLAG_DOCUMENT_21: 0.033%

Columnas eliminadas : 14
Shape actual        : (307511, 80)


---
## 3. Eliminación: FLAG_MOBIL y FLAG_CONT_MOBILE

**Criterio: varianza prácticamente cero**

- `FLAG_MOBIL`: 99.9997% de los registros tienen valor 1. Solo 1 registro tiene 0.
- `FLAG_CONT_MOBILE`: 99.8% en 1.

Una variable constante no puede discriminar entre clientes buenos y malos.  
Cualquier modelo la ignorará automáticamente, pero ocupa memoria y tiempo de entrenamiento.

**Decisión:** eliminar ambas columnas.

In [5]:
# Verificar antes de eliminar
print('Distribución antes de eliminar:')
print(f'  FLAG_MOBIL       : {app["FLAG_MOBIL"].value_counts().to_dict()}')
print(f'  FLAG_CONT_MOBILE : {app["FLAG_CONT_MOBILE"].value_counts().to_dict()}')

zero_var = ['FLAG_MOBIL', 'FLAG_CONT_MOBILE']
app = app.drop(columns=zero_var)
log_eliminaciones.append({'grupo': 'Varianza casi cero', 'n': len(zero_var), 'criterio': 'FLAG_MOBIL y FLAG_CONT_MOBILE >99.8% en 1'})

print(f'\nColumnas eliminadas : {len(zero_var)}')
print(f'Shape actual        : {app.shape}')

Distribución antes de eliminar:
  FLAG_MOBIL       : {1: 307510, 0: 1}
  FLAG_CONT_MOBILE : {1: 306937, 0: 574}

Columnas eliminadas : 2
Shape actual        : (307511, 78)


---
## 4. Eliminación: REGION_RATING_CLIENT

**Criterio: redundante con versión más granular**

`REGION_RATING_CLIENT` y `REGION_RATING_CLIENT_W_CITY` miden el rating de riesgo de la región donde vive el cliente.  
La diferencia es que `W_CITY` incorpora la ciudad además de la región — es más granular y más informativa.  
La correlación entre ambas es 0.95.

**Decisión:** conservar `REGION_RATING_CLIENT_W_CITY` y eliminar `REGION_RATING_CLIENT`.

In [6]:
corr = app[['REGION_RATING_CLIENT', 'REGION_RATING_CLIENT_W_CITY']].corr().iloc[0,1]
print(f'Correlación entre ambas: {corr:.4f}')

app = app.drop(columns=['REGION_RATING_CLIENT'])
log_eliminaciones.append({'grupo': 'Redundante por correlación', 'n': 1, 'criterio': 'REGION_RATING_CLIENT — correlación 0.95 con W_CITY'})

print(f'Columnas eliminadas : 1')
print(f'Shape actual        : {app.shape}')

Correlación entre ambas: 0.9508
Columnas eliminadas : 1
Shape actual        : (307511, 77)


---
## 5. Eliminación: OBS_30_CNT_SOCIAL_CIRCLE

**Criterio: redundante — correlación 0.998 con OBS_60**

`OBS_30_CNT_SOCIAL_CIRCLE` cuenta cuántas personas del círculo social del cliente fueron observadas en mora de 30 días.  
`OBS_60_CNT_SOCIAL_CIRCLE` hace lo mismo pero para mora de 60 días — e incluye a todos los que están en 30 también.  
La correlación entre ambas es 0.998 — prácticamente idénticas.

**Decisión:** conservar `OBS_60` (más informativa) y eliminar `OBS_30`.

In [7]:
corr = app[['OBS_30_CNT_SOCIAL_CIRCLE', 'OBS_60_CNT_SOCIAL_CIRCLE']].corr().iloc[0,1]
print(f'Correlación entre ambas: {corr:.4f}')

app = app.drop(columns=['OBS_30_CNT_SOCIAL_CIRCLE'])
log_eliminaciones.append({'grupo': 'Redundante por correlación', 'n': 1, 'criterio': 'OBS_30 — correlación 0.998 con OBS_60'})

print(f'Columnas eliminadas : 1')
print(f'Shape actual        : {app.shape}')

Correlación entre ambas: 0.9985
Columnas eliminadas : 1
Shape actual        : (307511, 76)


---
## 6. Corrección: DAYS_EMPLOYED = 365243

**Criterio: valor anómalo — código de sistema que no representa datos reales**

`DAYS_EMPLOYED` indica cuántos días lleva empleado el cliente (en negativo).  
Hay 55,374 registros (18%) con el valor `365243`, que equivale a ~1000 años — imposible en la realidad.  
Este valor es un código que el sistema usó para marcar clientes desempleados o sin dato de empleo.

Si no se corrige, contamina cualquier estadística de empleo y cualquier feature derivado.

**Decisión:**  
1. Crear `DAYS_EMPLOYED_FLAG = 1` para marcar esos registros (la señal de "desempleado" es valiosa).  
2. Reemplazar el valor anómalo por `NaN`.  
3. Imputar con la mediana de los valores válidos.

In [8]:
print(f'Registros con DAYS_EMPLOYED = 365243: {(app["DAYS_EMPLOYED"] == 365243).sum():,} ({(app["DAYS_EMPLOYED"] == 365243).mean()*100:.1f}%)')

app['DAYS_EMPLOYED_FLAG'] = (app['DAYS_EMPLOYED'] == 365243).astype(int)
app['DAYS_EMPLOYED']      = app['DAYS_EMPLOYED'].replace(365243, np.nan)
mediana_emp               = app['DAYS_EMPLOYED'].median()
app['DAYS_EMPLOYED']      = app['DAYS_EMPLOYED'].fillna(mediana_emp)

print(f'DAYS_EMPLOYED_FLAG creado  : {app["DAYS_EMPLOYED_FLAG"].sum():,} marcados como desempleados')
print(f'Imputado con mediana       : {mediana_emp:.0f} días')
print(f'Nulos restantes            : {app["DAYS_EMPLOYED"].isnull().sum()}')

Registros con DAYS_EMPLOYED = 365243: 55,374 (18.0%)
DAYS_EMPLOYED_FLAG creado  : 55,374 marcados como desempleados
Imputado con mediana       : -1648 días
Nulos restantes            : 0


---
## 7. Corrección: anomalías en variables categóricas

**Criterio: categorías espurias con muy pocos casos**

- `CODE_GENDER = 'XNA'`: 4 registros. No es un género válido — es un dato faltante codificado. Se reemplaza con la moda (F).
- `NAME_FAMILY_STATUS = 'Unknown'`: 2 registros. Mismo caso. Se reemplaza con la moda (Married).
- `CNT_CHILDREN > 14`: 7 registros con valores 14 y 19. Probables errores de carga. Se capea en 5.

Con tan pocos casos, estas categorías no pueden aportar señal y podrían crear problemas en encoding.

In [9]:
# CODE_GENDER
moda_gender = app[app['CODE_GENDER'] != 'XNA']['CODE_GENDER'].mode()[0]
app['CODE_GENDER'] = app['CODE_GENDER'].replace('XNA', moda_gender)
print(f'CODE_GENDER XNA reemplazado con: {moda_gender}')

# NAME_FAMILY_STATUS
moda_family = app[app['NAME_FAMILY_STATUS'] != 'Unknown']['NAME_FAMILY_STATUS'].mode()[0]
app['NAME_FAMILY_STATUS'] = app['NAME_FAMILY_STATUS'].replace('Unknown', moda_family)
print(f'NAME_FAMILY_STATUS Unknown reemplazado con: {moda_family}')

# CNT_CHILDREN
print(f'CNT_CHILDREN antes del cap — valores > 5: {(app["CNT_CHILDREN"] > 5).sum()}')
app['CNT_CHILDREN'] = app['CNT_CHILDREN'].clip(upper=5)
print(f'CNT_CHILDREN capeado en 5')

CODE_GENDER XNA reemplazado con: F
NAME_FAMILY_STATUS Unknown reemplazado con: Married
CNT_CHILDREN antes del cap — valores > 5: 42
CNT_CHILDREN capeado en 5


---
## 8. Imputación: EXT_SOURCE_1, EXT_SOURCE_2, EXT_SOURCE_3

**Criterio: predictores más fuertes del dataset — no se descartan por nulos**

Los tres scores externos tienen las correlaciones más altas con el TARGET (-0.155, -0.160, -0.179).  
Sus nulos no significan que el concepto no aplica — simplemente el score no estaba disponible en ese momento.

**Decisión:** imputar con la **mediana** de cada columna.  
Se usa mediana en lugar de media porque las distribuciones son asimétricas y la mediana es más robusta a outliers.

> `EXT_SOURCE_1` tiene 56% de nulos — mucho, pero la señal es tan fuerte que vale la pena conservarla.

In [10]:
for col in ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']:
    nulos_antes = app[col].isnull().sum()
    mediana     = app[col].median()
    app[col]    = app[col].fillna(mediana)
    print(f'{col}: {nulos_antes:,} nulos imputados con mediana = {mediana:.4f}')

EXT_SOURCE_1: 173,378 nulos imputados con mediana = 0.5060
EXT_SOURCE_2: 660 nulos imputados con mediana = 0.5660
EXT_SOURCE_3: 60,965 nulos imputados con mediana = 0.5353


---
## 9. Imputación: bloque housing _AVG

**Criterio: nulos no aleatorios — imputar con -1 como categoría 'no aplica'**

Las columnas `_AVG` del bloque housing tienen entre 48% y 70% de nulos.  
Este patrón no es aleatorio — corresponde a clientes que no viven en edificios con esas características (sin área común, sin ascensor, etc.).  

Imputar con la mediana sería incorrecto porque introduciría valores plausibles donde el concepto no existe.  
El -1 actúa como "no aplica" — un árbol de decisión puede aprender a tratarlo como categoría separada.

In [11]:
housing_avg_cols = [c for c in app.columns
                    if any(c.startswith(b) for b in housing_base) and c.endswith('_AVG')]

for c in housing_avg_cols:
    app[c] = app[c].fillna(-1)

# Categóricas del bloque
housing_cat = ['FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE',
               'EMERGENCYSTATE_MODE']
for c in housing_cat:
    if c in app.columns:
        app[c] = app[c].fillna('Unknown')

if 'TOTALAREA_MODE' in app.columns:
    app['TOTALAREA_MODE'] = app['TOTALAREA_MODE'].fillna(-1)

print(f'Columnas housing _AVG imputadas con -1 : {len(housing_avg_cols)}')
print(f'Columnas housing categóricas con Unknown: {len(housing_cat)}')

Columnas housing _AVG imputadas con -1 : 14
Columnas housing categóricas con Unknown: 4


---
## 10. Imputación: AMT_REQ_CREDIT_BUREAU_*

**Criterio: ausencia de consulta = 0 consultas**

Estas 6 columnas registran cuántas veces consultaron el historial crediticio del cliente en distintos períodos.  
Un nulo significa que no hay registro de consultas — lo que equivale a 0 consultas, no a un dato desconocido.

In [12]:
bureau_req_cols = [
    'AMT_REQ_CREDIT_BUREAU_HOUR', 'AMT_REQ_CREDIT_BUREAU_DAY',
    'AMT_REQ_CREDIT_BUREAU_WEEK', 'AMT_REQ_CREDIT_BUREAU_MON',
    'AMT_REQ_CREDIT_BUREAU_QRT',  'AMT_REQ_CREDIT_BUREAU_YEAR'
]
for c in bureau_req_cols:
    nulos = app[c].isnull().sum()
    app[c] = app[c].fillna(0)
    print(f'{c}: {nulos:,} nulos → 0')

AMT_REQ_CREDIT_BUREAU_HOUR: 41,519 nulos → 0
AMT_REQ_CREDIT_BUREAU_DAY: 41,519 nulos → 0
AMT_REQ_CREDIT_BUREAU_WEEK: 41,519 nulos → 0
AMT_REQ_CREDIT_BUREAU_MON: 41,519 nulos → 0
AMT_REQ_CREDIT_BUREAU_QRT: 41,519 nulos → 0
AMT_REQ_CREDIT_BUREAU_YEAR: 41,519 nulos → 0


---
## 11. Imputación: nulos pequeños (<1%)

**Criterio: pocos nulos, imputación estándar sin riesgo de sesgo significativo**

- `OCCUPATION_TYPE` (31%): ausencia puede ser señal — imputar con `'Unknown'`.
- `NAME_TYPE_SUITE` (0.42%), `OBS/DEF social circle` (0.33%), `AMT_GOODS_PRICE` (0.09%), `AMT_ANNUITY`, `CNT_FAM_MEMBERS`, `DAYS_LAST_PHONE_CHANGE`: imputar con mediana o moda según tipo.

In [13]:
# OCCUPATION_TYPE — 31% nulos, ausencia informativa
app['OCCUPATION_TYPE'] = app['OCCUPATION_TYPE'].fillna('Unknown')
print('OCCUPATION_TYPE: nulos → Unknown')

# NAME_TYPE_SUITE — categórica, moda
app['NAME_TYPE_SUITE'] = app['NAME_TYPE_SUITE'].fillna(app['NAME_TYPE_SUITE'].mode()[0])
print('NAME_TYPE_SUITE: nulos → moda')

# Social circle — mediana
for c in ['OBS_60_CNT_SOCIAL_CIRCLE', 'DEF_30_CNT_SOCIAL_CIRCLE', 'DEF_60_CNT_SOCIAL_CIRCLE']:
    app[c] = app[c].fillna(app[c].median())
print('Social circle: nulos → mediana')

# Montos y conteos — mediana
for c in ['AMT_GOODS_PRICE', 'AMT_ANNUITY', 'CNT_FAM_MEMBERS', 'DAYS_LAST_PHONE_CHANGE']:
    app[c] = app[c].fillna(app[c].median())
print('AMT_GOODS_PRICE, AMT_ANNUITY, CNT_FAM_MEMBERS, DAYS_LAST_PHONE_CHANGE: nulos → mediana')

# OWN_CAR_AGE — nulos = no tiene auto
app['OWN_CAR_AGE'] = app['OWN_CAR_AGE'].fillna(0)
print('OWN_CAR_AGE: nulos → 0 (no tiene auto)')

print(f'\nNulos en numéricos restantes: {app.select_dtypes(include=["float64","int64"]).isnull().sum().sum()}')

OCCUPATION_TYPE: nulos → Unknown
NAME_TYPE_SUITE: nulos → moda
Social circle: nulos → mediana
AMT_GOODS_PRICE, AMT_ANNUITY, CNT_FAM_MEMBERS, DAYS_LAST_PHONE_CHANGE: nulos → mediana
OWN_CAR_AGE: nulos → 0 (no tiene auto)

Nulos en numéricos restantes: 0


---
## 12. Feature engineering

**Columnas nuevas construidas a partir de las existentes**

| Feature | Lógica | Justificación |
|---|---|---|
| `AGE_YEARS` | `DAYS_BIRTH / -365.25` | Más interpretable que días negativos |
| `YEARS_EMPLOYED` | `DAYS_EMPLOYED / -365.25` | Más interpretable que días negativos |
| `CREDIT_INCOME_RATIO` | `AMT_CREDIT / AMT_INCOME_TOTAL` | Mide apalancamiento |
| `ANNUITY_INCOME_RATIO` | `AMT_ANNUITY / AMT_INCOME_TOTAL` | Carga de la cuota sobre ingreso |
| `CREDIT_GOODS_RATIO` | `AMT_CREDIT / AMT_GOODS_PRICE` | Qué tanto financia el banco |
| `INCOME_PER_PERSON` | `AMT_INCOME_TOTAL / CNT_FAM_MEMBERS` | Ingreso real disponible por persona |
| `EXT_SOURCE_MEAN` | Promedio de EXT_SOURCE 1, 2, 3 | Combinación — correlación 0.222 con TARGET |
| `EXT_SOURCE_MIN` | Mínimo de EXT_SOURCE 1, 2, 3 | Captura el peor score externo |
| `BUREAU_QUERIES_TOTAL` | Suma de AMT_REQ_CREDIT_BUREAU_* | Credit shopping — señal de búsqueda desesperada |

In [14]:
# Edad y empleo en años
app['AGE_YEARS']     = app['DAYS_BIRTH']    / -365.25
app['YEARS_EMPLOYED']= app['DAYS_EMPLOYED'] / -365.25

# Ratios financieros
app['CREDIT_INCOME_RATIO']  = app['AMT_CREDIT'] / app['AMT_INCOME_TOTAL']
app['ANNUITY_INCOME_RATIO'] = app['AMT_ANNUITY'] / app['AMT_INCOME_TOTAL']
app['CREDIT_GOODS_RATIO']   = app['AMT_CREDIT']  / app['AMT_GOODS_PRICE'].replace(0, np.nan)
app['CREDIT_GOODS_RATIO']   = app['CREDIT_GOODS_RATIO'].fillna(app['CREDIT_GOODS_RATIO'].median())

# Ingreso per cápita
app['INCOME_PER_PERSON'] = app['AMT_INCOME_TOTAL'] / app['CNT_FAM_MEMBERS'].replace(0, np.nan)
app['INCOME_PER_PERSON'] = app['INCOME_PER_PERSON'].fillna(app['INCOME_PER_PERSON'].median())

# EXT_SOURCE combinado
ext_cols = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
app['EXT_SOURCE_MEAN'] = app[ext_cols].mean(axis=1)
app['EXT_SOURCE_MIN']  = app[ext_cols].min(axis=1)

# Total consultas bureau
app['BUREAU_QUERIES_TOTAL'] = app[bureau_req_cols].sum(axis=1)

# DAYS_BIRTH ya fue reemplazado por AGE_YEARS
app = app.drop(columns=['DAYS_BIRTH'])

print('Features engineered creadas: AGE_YEARS, YEARS_EMPLOYED, CREDIT_INCOME_RATIO,')
print('  ANNUITY_INCOME_RATIO, CREDIT_GOODS_RATIO, INCOME_PER_PERSON,')
print('  EXT_SOURCE_MEAN, EXT_SOURCE_MIN, BUREAU_QUERIES_TOTAL')
print(f'\nDAYS_BIRTH eliminado (reemplazado por AGE_YEARS)')

Features engineered creadas: AGE_YEARS, YEARS_EMPLOYED, CREDIT_INCOME_RATIO,
  ANNUITY_INCOME_RATIO, CREDIT_GOODS_RATIO, INCOME_PER_PERSON,
  EXT_SOURCE_MEAN, EXT_SOURCE_MIN, BUREAU_QUERIES_TOTAL

DAYS_BIRTH eliminado (reemplazado por AGE_YEARS)


---
## Resumen final

In [15]:
print('=' * 55)
print('RESUMEN DE LIMPIEZA — application_train')
print('=' * 55)
print(f'Columnas originales          : {cols_originales}')
print()
print('ELIMINACIONES:')
total_drop = 0
for entry in log_eliminaciones:
    print(f"  {entry['grupo']:<35} -{entry['n']:>3}  ({entry['criterio']})")
    total_drop += entry['n']
# DAYS_BIRTH eliminado en paso 12
print(f"  {'DAYS_BIRTH (→ AGE_YEARS)':<35} -  1  (reemplazada por feature engineered)")
total_drop += 1
print(f'  {"-"*52}')
print(f'  Total eliminadas             : -{total_drop}')
print()
print('FEATURES NUEVAS (engineered)  : +9')
print()
print('=' * 55)
print(f'COLUMNAS FINALES             : {app.shape[1]}')
print(f'FILAS                        : {app.shape[0]:,}')
print(f'NULOS RESTANTES (numéricos)  : {app.select_dtypes(include=["float64","int64"]).isnull().sum().sum()}')
print('=' * 55)

RESUMEN DE LIMPIEZA — application_train
Columnas originales          : 122

ELIMINACIONES:
  Housing MODE y MEDI                 - 28  (Correlación >0.97 con _AVG)
  FLAG_DOCUMENT tasa <0.5%            - 14  (Varianza insuficiente)
  Varianza casi cero                  -  2  (FLAG_MOBIL y FLAG_CONT_MOBILE >99.8% en 1)
  Redundante por correlación          -  1  (REGION_RATING_CLIENT — correlación 0.95 con W_CITY)
  Redundante por correlación          -  1  (OBS_30 — correlación 0.998 con OBS_60)
  DAYS_BIRTH (→ AGE_YEARS)            -  1  (reemplazada por feature engineered)
  ----------------------------------------------------
  Total eliminadas             : -47

FEATURES NUEVAS (engineered)  : +9

COLUMNAS FINALES             : 85
FILAS                        : 307,511
NULOS RESTANTES (numéricos)  : 0


---
## 13. Exploración de Categorias de la tabla NAME_INCOME_TYPE

In [19]:
# NAME_INCOME_TYPE
conteo_income = app["NAME_INCOME_TYPE"].value_counts(dropna=False)

tabla_income = pd.DataFrame({
    "categoria": conteo_income.index,
    "cantidad": conteo_income.values,
    "porcentaje": (conteo_income.values / len(app) * 100).round(4)
})

tabla_income

,categoria,cantidad,porcentaje
0,Working,158774,51.6320
1,Commercial associate,71617,23.2892
2,Pensioner,55362,18.0033
3,State servant,21703,7.0576
4,Unemployed,22,0.0072
5,Student,18,0.0059
6,Businessman,10,0.0033
7,Maternity leave,5,0.0016


---
## 14. Agrupación de categorías pequeñas

**Criterio: Muy pocos casos de esas categorías**

- Tiene 4 categorías con menos de 50 casos cada una — Unemployed (22), Student (18), Businessman (10), Maternity leave (5). Son tan pocas que el modelo no puede aprender nada de ellas.

**Decisión: agruparlas en 'Other' antes de hacer encoding.**

In [ ]:

raras_income = ['Unemployed', 'Student', 'Businessman', 'Maternity leave']
app['NAME_INCOME_TYPE'] = app['NAME_INCOME_TYPE'].replace(raras_income, 'Other')

conteo_org = app['ORGANIZATION_TYPE'].value_counts()
raras_org  = conteo_org[conteo_org < 50].index.tolist()
app['ORGANIZATION_TYPE'] = app['ORGANIZATION_TYPE'].replace(raras_org, 'Other')

### 3. Outliers en variables numéricas

`AMT_INCOME_TOTAL` tiene 3.014 valores sobre el percentil 99 (`472.500`). No se consideran errores, ya que pueden corresponder a clientes con ingresos altos reales. Sin embargo, en los ratios construidos, como `CREDIT_INCOME_RATIO` y `ANNUITY_INCOME_RATIO`, un ingreso extremadamente alto puede generar valores cercanos a cero y distorsionar la distribución.

**Decisión:** no eliminar estos valores, pero capear los ratios derivados al percentil 99 para evitar que los extremos distorsionen el modelo.

In [24]:
for col in ['CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO', 'INCOME_PER_PERSON']:
    cap = app[col].quantile(0.99)
    app[col] = app[col].clip(upper=cap)